# LLM Pipeline Test — Step 6: Tool Optimization

The **Optimizer** is decoupled from gameplay.  This notebook orchestrates:

1. **Play** a Sokoban game with the MCTS engine → generate trace
2. **Collect** tool sources from the engine
3. **Optimise** — feed traces + tool sources to the Optimizer (LLM analysis → code generation → parse / validate / install → smoke test)
4. **Evaluate** — hot-swap the returned function into a fresh engine

We use **level3** (2 boxes) since default MCTS cannot solve it — the ideal test case for LLM improvement.

## Cell 1: Setup & imports

In [1]:
import sys
from pathlib import Path

# Ensure Tool_Creation is on sys.path
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mcts import MCTSEngine
from mcts.games import Sokoban
from LLM import Optimizer

# -- Configuration (single place to tweak) --
GAME      = "sokoban"
LEVEL     = "level3"
PHASE     = "simulation"        # MCTS phase to improve
ITERS     = 200                 # MCTS iterations per move
MAX_DEPTH = 500                 # max rollout depth
MAX_STEPS = 200                 # max game steps

print(f"Working dir: {ROOT}")
print("All imports OK ✓")

Working dir: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation
All imports OK ✓


## Cell 2: Play baseline game → collect trace & tool sources

The engine produces the trace file and exposes the current tool sources.
The Optimizer never touches the engine.

In [2]:
# Play one baseline game
game = Sokoban(LEVEL, max_steps=MAX_STEPS)
engine = MCTSEngine(game, iterations=ITERS, max_rollout_depth=MAX_DEPTH, logging=True)
baseline = engine.play_game()

# Collect outputs for the optimizer
trace_file  = baseline.get("log_file", "")
tool_list   = engine.get_tool_source()          # dict[str, str]

base_tag = "SOLVED" if baseline.get("solved") else "UNSOLVED"
print(f"Baseline: {base_tag} in {baseline.get('steps', '?')} steps  "
      f"returns={baseline.get('returns', '?')}")
print(f"Trace: {trace_file}")
print(f"Tools: {list(tool_list.keys())}")

Baseline: UNSOLVED in 200 steps  returns=[0.1714285714285714]
Trace: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/mcts/records/Sokoban (level3)_20260306_002414_173661.json
Tools: ['selection', 'expansion', 'simulation', 'backpropagation']


## Cell 3: Iterative Optimization Loop (5 iterations)

Each iteration:
1. **Play** a game using the *best-so-far* tool → new trace
2. **Build history context** from all previous iterations' results
3. **Run optimizer** with trace + tool sources + history context
4. **Evaluate** the new tool with **3 runs** (reduce variance)
5. **Keep best** — only adopt the new tool if it beats the current best

Fixes vs. previous version:
- **Multi-run eval** (3 games) to reduce MCTS variance
- **Best-so-far gating** — never regress to a worse function
- **Consistent filename** — always `simulation.py` to avoid clutter

In [3]:
NUM_ITERS  = 5
EVAL_RUNS  = 3          # average over this many games per evaluation

opt = Optimizer(
    game=GAME,
    target_phase=PHASE,
    two_step=True,
    verbose=True,
)

# State that carries across iterations
best_fn      = None        # best-so-far function (None = default)
best_score   = 0.0         # best-so-far average returns
current_fn   = None        # latest function (may equal best_fn)
all_results  = []
state_factory = lambda: Sokoban(LEVEL, max_steps=MAX_STEPS).new_initial_state()


def multi_eval(fn, n=EVAL_RUNS):
    """Evaluate a function over n games; return (avg_returns, solve_rate, results)."""
    results = []
    for _ in range(n):
        g = Sokoban(LEVEL, max_steps=MAX_STEPS)
        e = MCTSEngine(g, iterations=ITERS, max_rollout_depth=MAX_DEPTH, logging=True)
        if fn is not None:
            e.set_tool(PHASE, fn)
        r = e.play_game()
        results.append(r)
    avg_ret = sum(r["returns"][0] if isinstance(r["returns"], list)
                  else r["returns"] for r in results) / n
    solve_rate = sum(1 for r in results if r.get("solved")) / n
    return avg_ret, solve_rate, results


def build_history(results, baseline_info):
    """Build a concise history string from previous iterations."""
    lines = [
        f"Baseline (default MCTS): avg_returns={baseline_info['avg_returns']:.4f}, "
        f"solve_rate={baseline_info['solve_rate']:.0%}, "
        f"steps={baseline_info['avg_steps']:.0f}",
        f"Current best: avg_returns={best_score:.4f}",
    ]
    for r in results:
        tag = f"solve_rate={r['solve_rate']:.0%}"
        adopted = " ← ADOPTED" if r.get("adopted") else ""
        lines.append(
            f"  Iter {r['iteration']}: avg_returns={r['avg_returns']:.4f}, "
            f"{tag}, desc={r.get('description', 'n/a')}{adopted}"
        )
        if r.get("analysis_snippet"):
            lines.append(f"    Insight: {r['analysis_snippet']}")
    return "\n".join(lines)


# --- Baseline multi-eval ---
print("Evaluating baseline (default MCTS)…")
base_avg, base_solve, base_runs = multi_eval(None, n=EVAL_RUNS)
baseline_info = {
    "avg_returns": base_avg,
    "solve_rate": base_solve,
    "avg_steps": sum(r.get("steps", MAX_STEPS) for r in base_runs) / EVAL_RUNS,
}
print(f"  Baseline: avg_returns={base_avg:.4f}, solve_rate={base_solve:.0%}")
best_score = base_avg  # anything better than this gets adopted


for iteration in range(1, NUM_ITERS + 1):
    print(f"\n{'#'*60}")
    print(f"  ITERATION {iteration}/{NUM_ITERS}")
    print(f"{'#'*60}")

    # --- 1. Play with best-so-far tool → collect trace ---
    g = Sokoban(LEVEL, max_steps=MAX_STEPS)
    eng = MCTSEngine(g, iterations=ITERS, max_rollout_depth=MAX_DEPTH, logging=True)
    if best_fn is not None:
        eng.set_tool(PHASE, best_fn)

    play_result = eng.play_game()
    play_trace  = play_result.get("log_file", "")
    tl          = eng.get_tool_source()
    p_ret = play_result["returns"]
    p_ret_val = p_ret[0] if isinstance(p_ret, list) else p_ret
    ptag = "SOLVED" if play_result.get("solved") else "UNSOLVED"
    print(f"  Play: {ptag} in {play_result.get('steps','?')} steps  "
          f"returns={p_ret_val:.4f}")

    # --- 2. Build history context ---
    history = build_history(all_results, baseline_info) if all_results else None

    # --- 3. Optimize ---
    result = opt.run(
        record_files=[play_trace] if play_trace else [],
        tool_list=tl,
        state_factory=state_factory,
        additional_context=history,
    )

    # --- 4. Multi-run evaluation ---
    iter_record = {
        "iteration": iteration,
        "smoke_test": result["smoke_test"],
        "avg_returns": base_avg,
        "solve_rate": 0.0,
        "steps": MAX_STEPS,
        "description": (result.get("parsed") or {}).get("description", ""),
        "error": result.get("error"),
        "analysis_snippet": "",
        "adopted": False,
    }

    analysis = (result.get("llm_result") or {}).get("step1_analysis", "")
    if analysis:
        iter_record["analysis_snippet"] = analysis[:200].replace("\n", " ")

    fn = result.get("function")
    if fn is not None:
        avg_ret, solve_rate, eval_runs = multi_eval(fn, n=EVAL_RUNS)
        avg_steps = sum(r.get("steps", MAX_STEPS) for r in eval_runs) / EVAL_RUNS
        iter_record["avg_returns"] = avg_ret
        iter_record["solve_rate"] = solve_rate
        iter_record["steps"] = avg_steps

        etag = f"solve_rate={solve_rate:.0%}"
        print(f"  Eval ({EVAL_RUNS} runs): avg_returns={avg_ret:.4f}, {etag}, "
              f"avg_steps={avg_steps:.0f}")

        # --- 5. Keep only if better ---
        if avg_ret > best_score:
            print(f"  ★ NEW BEST (prev best={best_score:.4f}) — adopting")
            best_fn = fn
            best_score = avg_ret
            iter_record["adopted"] = True
        else:
            print(f"  ✗ Not better than best ({best_score:.4f}) — keeping previous")
    else:
        print(f"  Eval:  SKIPPED (smoke test failed or error)")
        if result.get("error"):
            print(f"         {result['error'][:120]}")

    all_results.append(iter_record)

print(f"\n{'='*60}")
print(f"Iterative optimization complete — {NUM_ITERS} iterations done.")
print(f"Final best avg_returns={best_score:.4f}")
print(f"{'='*60}")

Evaluating baseline (default MCTS)…
  Baseline: avg_returns=0.1714, solve_rate=0%

############################################################
  ITERATION 1/5
############################################################
  Play: UNSOLVED in 200 steps  returns=0.6071
Step 1/4: Querying LLM (step 1 — analysis)…
Step 2/4: Querying LLM (step 2 — code generation)…
  LLM query: status=success  elapsed=41.67s
  Step-1 analysis length: 7999 chars
Step 3/4: Parsing & validating response…
  Validation passed ✓
  Installed to: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/simulation/simulation.py
Step 4/4: Smoke testing…
  Smoke test passed ✓
  Eval (3 runs): avg_returns=0.6071, solve_rate=0%, avg_steps=200
  ★ NEW BEST (prev best=0.1714) — adopting

############################################################
  ITERATION 2/5
############################################################
  Play: UNSOLVED in 200 steps  returns=0.6071
Step 1/4: Querying L

Task exception was never retrieved
future: <Task finished name='Task-47' coro=<AsyncClient.aclose() done, defined at /Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_client.py:1978> exception=RuntimeError('Event loop is closed')>
Traceback (most recent call last):
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_client.py", line 1985, in aclose
    await self._transport.aclose()
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_transports/default.py", line 406, in aclose
    await self._pool.aclose()
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 353, in aclose
    await self._close_connections(closing_connections)
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 345, in _close_connections
    await connection.aclose()
  File "/User

  LLM query: status=success  elapsed=34.78s
  Step-1 analysis length: 8459 chars
Step 3/4: Parsing & validating response…
  Validation passed ✓
  Installed to: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/simulation/simulation.py
Step 4/4: Smoke testing…
  Smoke test passed ✓
  Eval (3 runs): avg_returns=0.5786, solve_rate=0%, avg_steps=200
  ✗ Not better than best (0.6071) — keeping previous

############################################################
  ITERATION 3/5
############################################################
  Play: UNSOLVED in 200 steps  returns=0.6071
Step 1/4: Querying LLM (step 1 — analysis)…
Step 2/4: Querying LLM (step 2 — code generation)…


Task exception was never retrieved
future: <Task finished name='Task-57' coro=<AsyncClient.aclose() done, defined at /Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_client.py:1978> exception=RuntimeError('Event loop is closed')>
Traceback (most recent call last):
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_client.py", line 1985, in aclose
    await self._transport.aclose()
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_transports/default.py", line 406, in aclose
    await self._pool.aclose()
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 353, in aclose
    await self._close_connections(closing_connections)
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 345, in _close_connections
    await connection.aclose()
  File "/User

  LLM query: status=success  elapsed=36.93s
  Step-1 analysis length: 8103 chars
Step 3/4: Parsing & validating response…
  Validation passed ✓
  Installed to: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/simulation/simulation.py
Step 4/4: Smoke testing…
  Smoke test passed ✓
  Eval (3 runs): avg_returns=0.6071, solve_rate=0%, avg_steps=200
  ✗ Not better than best (0.6071) — keeping previous

############################################################
  ITERATION 4/5
############################################################
  Play: UNSOLVED in 200 steps  returns=0.6071
Step 1/4: Querying LLM (step 1 — analysis)…
Step 2/4: Querying LLM (step 2 — code generation)…


Task exception was never retrieved
future: <Task finished name='Task-67' coro=<AsyncClient.aclose() done, defined at /Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_client.py:1978> exception=RuntimeError('Event loop is closed')>
Traceback (most recent call last):
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_client.py", line 1985, in aclose
    await self._transport.aclose()
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_transports/default.py", line 406, in aclose
    await self._pool.aclose()
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 353, in aclose
    await self._close_connections(closing_connections)
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 345, in _close_connections
    await connection.aclose()
  File "/User

  LLM query: status=success  elapsed=48.08s
  Step-1 analysis length: 8671 chars
Step 3/4: Parsing & validating response…
  Validation passed ✓
  Installed to: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/simulation/simulation.py
Step 4/4: Smoke testing…
  Smoke test passed ✓
  Eval (3 runs): avg_returns=1.0000, solve_rate=100%, avg_steps=31
  ★ NEW BEST (prev best=0.6071) — adopting

############################################################
  ITERATION 5/5
############################################################
  Play: SOLVED in 154 steps  returns=1.0000
Step 1/4: Querying LLM (step 1 — analysis)…
Step 2/4: Querying LLM (step 2 — code generation)…


Task exception was never retrieved
future: <Task finished name='Task-77' coro=<AsyncClient.aclose() done, defined at /Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_client.py:1978> exception=RuntimeError('Event loop is closed')>
Traceback (most recent call last):
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_client.py", line 1985, in aclose
    await self._transport.aclose()
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpx/_transports/default.py", line 406, in aclose
    await self._pool.aclose()
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 353, in aclose
    await self._close_connections(closing_connections)
  File "/Users/zhuyuezx/miniconda3/envs/CSE291A_26W/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 345, in _close_connections
    await connection.aclose()
  File "/User

  LLM query: status=success  elapsed=48.73s
  Step-1 analysis length: 9654 chars
Step 3/4: Parsing & validating response…
  Validation passed ✓
  Installed to: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/simulation/simulation.py
Step 4/4: Smoke testing…
  Smoke test passed ✓
  Eval (3 runs): avg_returns=0.5643, solve_rate=0%, avg_steps=200
  ✗ Not better than best (1.0000) — keeping previous

Iterative optimization complete — 5 iterations done.
Final best avg_returns=1.0000


## Cell 4: Results Summary

In [6]:
# ── pretty-print results table ──────────────────────────────────────
header = f"{'Iter':<6} {'Solve%':>7} {'AvgRet':>10} {'AvgSteps':>10} {'Adopted':>8}  Description"
print(header)
print("─" * len(header))

# baseline
b = baseline_info
print(f"{'base':<6} {b['solve_rate']*100:>6.0f}% {b['avg_returns']:>10.3f} {b['avg_steps']:>10.1f} {'   —':>8}  default random rollout")

# iterations
for i, r in enumerate(all_results):
    tag = "✓" if r.get("adopted") else "✗"
    desc = r.get("description", "")[:50]
    print(f"{i} {r['solve_rate']*100:>6.0f}% {r['avg_returns']:>10.3f} {tag:>8}  {desc}")

print()
print(f"Best score achieved: {best_score:.3f}")

Iter    Solve%     AvgRet   AvgSteps  Adopted  Description
──────────────────────────────────────────────────────────
base        0%      0.171      200.0        —  default random rollout
0      0%      0.607        ✓  Added ε‑greedy look‑ahead with dead‑lock pruning, 
1      0%      0.579        ✗  Added stronger dead‑lock detection, richer five‑te
2      0%      0.607        ✗  Refined rollout with balanced heuristic, relaxed d
3    100%      1.000        ✓  Added stronger dead‑lock detection, push‑biased ε‑
4      0%      0.564        ✗  Added stronger dead‑lock checks, re‑balanced heuri

Best score achieved: 1.000


## Cell 5: Inspect Best Tool Code

In [7]:
import inspect

if best_fn is not None:
    print(f"Best function: {best_fn.__name__}")
    print(f"Best score:    {best_score:.3f}")
    print()
    print(inspect.getsource(best_fn))
else:
    print("No improvement found over baseline — best_fn is still the default.")

Best function: default_simulation
Best score:    1.000

def default_simulation(state, perspective_player: int, max_depth: int = 1000) -> float:
    """
    Refined rollout for Sokoban.

    Main improvements:
      • Extended dead‑lock detection (corner, wall‑line, 2×2 clusters,
        frozen corridor, simple dead‑square detection).
      • Adaptive rollout horizon (longer when more boxes are already placed).
      • Lower ε (0.15) with *improving* push bias – random pushes are only
        taken if they reduce Manhattan distance to a target.
      • Re‑balanced 6‑term heuristic:
            0.45 – boxes on targets
            0.20 – normalised total Manhattan distance
            0.15 – fraction of boxes that have at least one reachable improving push
            0.10 – alignment (clear row/col with a free target)
            0.05 – player mobility (reachable area)
            0.05 – proximity to the nearest improving push cell
      • Scaled push‑bonus proportional to the distance r